# Set Up

In [2]:
import requests
import pandas as pd
import time

from google.cloud import bigquery

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

from datetime import datetime, timezone

from google import genai
from google.genai import types
import json

In [3]:
PROJECT_ID = "pacey32-agency"

BQ = bigquery.Client(project=PROJECT_ID)

geolocator = Nominatim(user_agent="pacey32_agency_geospatial")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1,
)

In [4]:
TEAM_SQL = """
SELECT
    org.id,
    org.fullName,
    org.tricode,
    org.venue,
    org.venueLocation,
    city.state_province,
    city.country,
    city.latitude AS city_latitude,
    city.longitude AS city_longitude
FROM `pacey32-agency.Team.OrganizationDetail` org
LEFT JOIN `pacey32-agency.City.CityReference` city
ON org.venueLocation = city.city_name
ORDER BY fullName
"""

teams = (
    BQ.query(TEAM_SQL)
      .result()
      .to_dataframe()
)

#display(teams)

In [5]:
def geocode_location(name, city, state, country):

    query = f"{name}, {city}, {state}, {country}"

    location = geocode(query)

    if location is None:
        return {
            "query": query,
            "latitude": None,
            "longitude": None,
            "matched_address": None,
        }

    return {
        "query": query,
        "latitude": location.latitude,
        "longitude": location.longitude,
        "matched_address": location.address,
    }

In [6]:
def qa(df, lat="latitude", lon="longitude"):

    print(f"Rows: {len(df)}")
    print(f"Missing coordinates: {df[lat].isna().sum()}")

    display(
        df[
            df[lat].isna()
            | df[lon].isna()
        ]
    )

# ARENA

In [7]:
arena_rows = []

for _, row in teams.iterrows():

    result = geocode_location(
        name=row.venue,
        city=row.venueLocation,
        state=row.state_province,
        country=row.country,
    )

    arena_rows.append({

        "id": row.id,
        "fullName": row.fullName,
        "tricode": row.tricode,

        "arena_name": row.venue,

        "latitude": result["latitude"],
        "longitude": result["longitude"],

        "matched_address": result["matched_address"],
        "query": result["query"],

        "source": "Nominatim",
        "last_updated": datetime.now(timezone.utc)

    })

arena_df = pd.DataFrame(arena_rows)

In [8]:
#display(arena_df)

#qa(arena_df)

#arena_df.sort_values("fullName")

In [9]:
arena_df = arena_df[
    [
        "id",
        "tricode",
        "fullName",
        "arena_name",
        "latitude",
        "longitude",
        "matched_address",
        "source",
        "last_updated",
    ]
]

#display(arena_df)

# PRACTISE FACILITIES

In [10]:
practice_lookup = [
    {
        "id": 24,
        "fullName": "Anaheim Ducks",
        "tricode": "ANA",
        "facility_name": "Great Park Ice & FivePoint Arena",
        "facility_type": "Dedicated practice facility",
        "address": "888 Ridge Valley, Irvine, CA 92618",
        "city": "Irvine",
        "state_province": "California",
        "country": "United States",
        "notes": "Official Ducks practice facility."
    },
    {
        "id": 6,
        "fullName": "Boston Bruins",
        "tricode": "BOS",
        "facility_name": "Warrior Ice Arena",
        "facility_type": "Dedicated practice facility",
        "address": "90 Guest Street, Boston, MA 02135",
        "city": "Boston",
        "state_province": "Massachusetts",
        "country": "United States",
        "notes": "Official Bruins practice facility."
    },
    {
        "id": 7,
        "fullName": "Buffalo Sabres",
        "tricode": "BUF",
        "facility_name": "LECOM Harborcenter",
        "facility_type": "Arena-connected",
        "address": "100 Washington Street, Buffalo, NY 14203",
        "city": "Buffalo",
        "state_province": "New York",
        "country": "United States",
        "notes": "Connected to KeyBank Center."
    },
    {
        "id": 20,
        "fullName": "Calgary Flames",
        "tricode": "CGY",
        "facility_name": "Scotiabank Saddledome",
        "facility_type": "Arena",
        "address": "555 Saddledome Rise SE, Calgary, AB",
        "city": "Calgary",
        "state_province": "Alberta",
        "country": "Canada",
        "notes": "Current practice location until Scotia Place opens."
    },
    {
        "id": 12,
        "fullName": "Carolina Hurricanes",
        "tricode": "CAR",
        "facility_name": "Wake Competition Center",
        "facility_type": "Dedicated practice facility",
        "address": "801 Corporate Center Drive, Raleigh, NC",
        "city": "Morrisville",
        "state_province": "North Carolina",
        "country": "United States",
        "notes": "Formerly Invisalign Arena."
    },
    {
        "id": 16,
        "fullName": "Chicago Blackhawks",
        "tricode": "CHI",
        "facility_name": "Fifth Third Arena",
        "facility_type": "Dedicated practice facility",
        "address": "1801 W Jackson Blvd, Chicago, IL 60612",
        "city": "Chicago",
        "state_province": "Illinois",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 21,
        "fullName": "Colorado Avalanche",
        "tricode": "COL",
        "facility_name": "Family Sports Center",
        "facility_type": "Dedicated practice facility",
        "address": "6901 S Peoria Street, Centennial, CO",
        "city": "Centennial",
        "state_province": "Colorado",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 29,
        "fullName": "Columbus Blue Jackets",
        "tricode": "CBJ",
        "facility_name": "OhioHealth Ice Haus",
        "facility_type": "Arena-connected",
        "address": "200 W Nationwide Blvd, Columbus, OH",
        "city": "Columbus",
        "state_province": "Ohio",
        "country": "United States",
        "notes": "Connected to Nationwide Arena."
    },
    {
        "id": 25,
        "fullName": "Dallas Stars",
        "tricode": "DAL",
        "facility_name": "Comerica Center",
        "facility_type": "Dedicated practice facility",
        "address": "2601 Avenue of the Stars, Frisco, TX",
        "city": "Frisco",
        "state_province": "Texas",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 17,
        "fullName": "Detroit Red Wings",
        "tricode": "DET",
        "facility_name": "BELFOR Training Center",
        "facility_type": "Arena-connected",
        "address": "2645 Woodward Avenue, Detroit, MI",
        "city": "Detroit",
        "state_province": "Michigan",
        "country": "United States",
        "notes": "Located within Little Caesars Arena."
    },
    {
        "id": 22,
        "fullName": "Edmonton Oilers",
        "tricode": "EDM",
        "facility_name": "Downtown Community Arena",
        "facility_type": "Arena-connected",
        "address": "10220 104 Avenue NW, Edmonton, AB",
        "city": "Edmonton",
        "state_province": "Alberta",
        "country": "Canada",
        "notes": "Connected to Rogers Place."
    },
    {
        "id": 13,
        "fullName": "Florida Panthers",
        "tricode": "FLA",
        "facility_name": "Baptist Health IcePlex",
        "facility_type": "Dedicated practice facility",
        "address": "800 NE 8th Street, Fort Lauderdale, FL",
        "city": "Fort Lauderdale",
        "state_province": "Florida",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 26,
        "fullName": "Los Angeles Kings",
        "tricode": "LAK",
        "facility_name": "Toyota Sports Performance Center",
        "facility_type": "Dedicated practice facility",
        "address": "555 N Nash Street, El Segundo, CA",
        "city": "El Segundo",
        "state_province": "California",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 30,
        "fullName": "Minnesota Wild",
        "tricode": "MIN",
        "facility_name": "TRIA Rink",
        "facility_type": "Dedicated practice facility",
        "address": "400 Wabasha Street N, St Paul, MN",
        "city": "St. Paul",
        "state_province": "Minnesota",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 8,
        "fullName": "Montréal Canadiens",
        "tricode": "MTL",
        "facility_name": "CN Sports Complex",
        "facility_type": "Dedicated practice facility",
        "address": "8000 Boulevard Leduc, Brossard, QC",
        "city": "Brossard",
        "state_province": "Quebec",
        "country": "Canada",
        "notes": ""
    },
    {
        "id": 18,
        "fullName": "Nashville Predators",
        "tricode": "NSH",
        "facility_name": "Ford Ice Center Bellevue",
        "facility_type": "Dedicated practice facility",
        "address": "7638 B Hwy 70 S, Nashville, TN",
        "city": "Nashville",
        "state_province": "Tennessee",
        "country": "United States",
        "notes": "Primary practice facility."
    },
    {
        "id": 1,
        "fullName": "New Jersey Devils",
        "tricode": "NJD",
        "facility_name": "RWJBarnabas Health Hockey House",
        "facility_type": "Arena-connected",
        "address": "25 Lafayette Street, Newark, NJ",
        "city": "Newark",
        "state_province": "New Jersey",
        "country": "United States",
        "notes": "Attached to Prudential Center."
    },
    {
        "id": 2,
        "fullName": "New York Islanders",
        "tricode": "NYI",
        "facility_name": "Northwell Health Ice Center",
        "facility_type": "Dedicated practice facility",
        "address": "200 Merrick Avenue, East Meadow, NY",
        "city": "East Meadow",
        "state_province": "New York",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 3,
        "fullName": "New York Rangers",
        "tricode": "NYR",
        "facility_name": "MSG Training Center",
        "facility_type": "Dedicated practice facility",
        "address": "600 Corporate Court, Greenburgh, NY",
        "city": "Greenburgh",
        "state_province": "New York",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 9,
        "fullName": "Ottawa Senators",
        "tricode": "OTT",
        "facility_name": "Bell Sensplex",
        "facility_type": "Dedicated practice facility",
        "address": "1565 Maple Grove Road, Ottawa, ON",
        "city": "Ottawa",
        "state_province": "Ontario",
        "country": "Canada",
        "notes": ""
    },
    {
        "id": 4,
        "fullName": "Philadelphia Flyers",
        "tricode": "PHI",
        "facility_name": "Flyers Training Center",
        "facility_type": "Dedicated practice facility",
        "address": "601 Laurel Oak Road, Voorhees, NJ",
        "city": "Voorhees",
        "state_province": "New Jersey",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 5,
        "fullName": "Pittsburgh Penguins",
        "tricode": "PIT",
        "facility_name": "UPMC Lemieux Sports Complex",
        "facility_type": "Dedicated practice facility",
        "address": "8000 Cranberry Springs Drive, Cranberry Township, PA",
        "city": "Cranberry Township",
        "state_province": "Pennsylvania",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 28,
        "fullName": "San Jose Sharks",
        "tricode": "SJS",
        "facility_name": "Sharks Ice at San Jose",
        "facility_type": "Dedicated practice facility",
        "address": "1500 S 10th Street, San Jose, CA",
        "city": "San Jose",
        "state_province": "California",
        "country": "United States",
        "notes": "Adjacent to Tech CU Arena."
    },
    {
        "id": 55,
        "fullName": "Seattle Kraken",
        "tricode": "SEA",
        "facility_name": "Kraken Community Iceplex",
        "facility_type": "Dedicated practice facility",
        "address": "10601 5th Avenue NE, Seattle, WA",
        "city": "Seattle",
        "state_province": "Washington",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 19,
        "fullName": "St. Louis Blues",
        "tricode": "STL",
        "facility_name": "Centene Community Ice Center",
        "facility_type": "Dedicated practice facility",
        "address": "750 Casino Center Drive, Maryland Heights, MO",
        "city": "Maryland Heights",
        "state_province": "Missouri",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 14,
        "fullName": "Tampa Bay Lightning",
        "tricode": "TBL",
        "facility_name": "TGH Ice Plex",
        "facility_type": "Dedicated practice facility",
        "address": "10222 Elizabeth Place, Tampa, FL",
        "city": "Brandon",
        "state_province": "Florida",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 10,
        "fullName": "Toronto Maple Leafs",
        "tricode": "TOR",
        "facility_name": "Ford Performance Centre",
        "facility_type": "Dedicated practice facility",
        "address": "400 Kipling Avenue, Toronto, ON",
        "city": "Toronto",
        "state_province": "Ontario",
        "country": "Canada",
        "notes": ""
    },
    {
        "id": 68,
        "fullName": "Utah Mammoth",
        "tricode": "UTA",
        "facility_name": "Utah Mammoth Ice Center",
        "facility_type": "Dedicated practice facility",
        "address": "Sandy, UT",
        "city": "Sandy",
        "state_province": "Utah",
        "country": "United States",
        "notes": "Temporary address until permanent facility details are published."
    },
    {
        "id": 23,
        "fullName": "Vancouver Canucks",
        "tricode": "VAN",
        "facility_name": "UBC Doug Mitchell Thunderbird Sports Centre",
        "facility_type": "Shared practice facility",
        "address": "6066 Thunderbird Boulevard, Vancouver, BC",
        "city": "Vancouver",
        "state_province": "British Columbia",
        "country": "Canada",
        "notes": "Primary practice venue."
    },
    {
        "id": 54,
        "fullName": "Vegas Golden Knights",
        "tricode": "VGK",
        "facility_name": "City National Arena",
        "facility_type": "Dedicated practice facility",
        "address": "1550 S Pavilion Center Drive, Las Vegas, NV",
        "city": "Las Vegas",
        "state_province": "Nevada",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 15,
        "fullName": "Washington Capitals",
        "tricode": "WSH",
        "facility_name": "MedStar Capitals Iceplex",
        "facility_type": "Dedicated practice facility",
        "address": "627 N Glebe Road, Arlington, VA",
        "city": "Arlington",
        "state_province": "Virginia",
        "country": "United States",
        "notes": ""
    },
    {
        "id": 52,
        "fullName": "Winnipeg Jets",
        "tricode": "WPG",
        "facility_name": "Hockey for All Centre",
        "facility_type": "Dedicated practice facility",
        "address": "3969 Portage Avenue, Winnipeg, MB",
        "city": "Winnipeg",
        "state_province": "Manitoba",
        "country": "Canada",
        "notes": ""
    },
]

practice_lookup_df = pd.DataFrame(practice_lookup)

#display(practice_lookup_df)

In [12]:
practice_lookup_df.head(20)

,id,fullName,tricode,facility_name,facility_type,address,city,state_province,country,notes
0,24,Anaheim Ducks,ANA,Great Park Ice & FivePoint Arena,Dedicated practice facility,"888 Ridge Valley, Irvine, CA 92618",Irvine,California,United States,Official Ducks practice facility.
1,6,Boston Bruins,BOS,Warrior Ice Arena,Dedicated practice facility,"90 Guest Street, Boston, MA 02135",Boston,Massachusetts,United States,Official Bruins practice facility.
2,7,Buffalo Sabres,BUF,LECOM Harborcenter,Arena-connected,"100 Washington Street, Buffalo, NY 14203",Buffalo,New York,United States,Connected to KeyBank Center.
3,20,Calgary Flames,CGY,Scotiabank Saddledome,Arena,"555 Saddledome Rise SE, Calgary, AB",Calgary,Alberta,Canada,Current practice location until Scotia Place o...
4,12,Carolina Hurricanes,CAR,Wake Competition Center,Dedicated practice facility,"801 Corporate Center Drive, Raleigh, NC",Morrisville,North Carolina,United States,Formerly Invisalign Arena.
5,16,Chicago Blackhawks,CHI,Fifth Third Arena,Dedicated practice facility,"1801 W Jackson Blvd, Chicago, IL 60612",Chicago,Illinois,United States,
6,21,Colorado Avalanche,COL,Family Sports Center,Dedicated practice facility,"6901 S Peoria Street, Centennial, CO",Centennial,Colorado,United States,
7,29,Columbus Blue Jackets,CBJ,OhioHealth Ice Haus,Arena-connected,"200 W Nationwide Blvd, Columbus, OH",Columbus,Ohio,United States,Connected to Nationwide Arena.
8,25,Dallas Stars,DAL,Comerica Center,Dedicated practice facility,"2601 Avenue of the Stars, Frisco, TX",Frisco,Texas,United States,
9,17,Detroit Red Wings,DET,BELFOR Training Center,Arena-connected,"2645 Woodward Avenue, Detroit, MI",Detroit,Michigan,United States,Located within Little Caesars Arena.


# POIs

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://api.geoapify.com/v2/places"
RADIUS_METRES = 50000

LIMIT = 10

In [59]:
POI_CATEGORIES = {
    "Airport": "airport",
    "Hospital": "healthcare.hospital",
    "School": "education.school",
    "Golf Course": "sport.golf_course",
    "Sports Club": "activity.sport_club",
    "Marina": "maritime.marina",
    "Ski": "ski",
    "Beach": "beach",
    "Shopping Mall": "commercial.shopping_mall",
    "Restaurant": "catering.restaurant",
}

In [60]:
def get_pois(
    latitude,
    longitude,
    category,
    radius=50000,
    limit=10,
):

    params = {
        "categories": category,
        "filter": f"circle:{longitude},{latitude},{radius}",
        "limit": limit,
        "apiKey": GEOAPIFY_API_KEY,
    }

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=60,
    )

    response.raise_for_status()

    return response.json()["features"]

In [62]:
import time
from datetime import datetime, timezone

poi_rows = []

total_teams = len(arena_df)

for team_number, (_, team) in enumerate(arena_df.iterrows(), start=1):

    print(f"\n[{team_number}/{total_teams}] {team.fullName}")

    for category_name, category in POI_CATEGORIES.items():

        print(f"   {category_name:<18}", end="")

        try:

            features = get_pois(
                latitude=team.latitude,
                longitude=team.longitude,
                category=category,
                radius=50000,
                limit=10,
            )

            print(f" {len(features)} found")

            for feature in features:

                props = feature["properties"]

                poi_rows.append({
                    "team_id": team.id,
                    "tricode": team.tricode,
                    "fullName": team.fullName,
                    "category": category_name,
                    "name": props.get("name"),
                    "address": props.get("formatted"),
                    "latitude": props.get("lat"),
                    "longitude": props.get("lon"),
                    "city": props.get("city"),
                    "state": props.get("state"),
                    "country": props.get("country"),
                    "last_updated": datetime.now(timezone.utc),
                    "source": "Geoapify",
                })

            # Small pause between API calls
            time.sleep(0.25)

        except Exception as ex:

            print(f" ERROR: {ex}")

poi_df = (
    pd.DataFrame(poi_rows)
    .sort_values(
        ["fullName", "category", "name"],
        ignore_index=True,
    )
)

display(poi_df)

print()
print(f"{len(poi_df):,} POIs collected")


[1/32] Anaheim Ducks
   Airport            10 found
   Hospital           10 found
   School             10 found
   Golf Course        10 found
   Sports Club        10 found
   Marina             10 found
   Ski                0 found
   Beach              10 found
   Shopping Mall      10 found
   Restaurant         10 found

[2/32] Boston Bruins
   Airport            10 found
   Hospital           10 found
   School             10 found
   Golf Course        10 found
   Sports Club        10 found
   Marina             10 found
   Ski                10 found
   Beach              10 found
   Shopping Mall      10 found
   Restaurant         10 found

[3/32] Buffalo Sabres
   Airport            7 found
   Hospital           10 found
   School             10 found
   Golf Course        10 found
   Sports Club        10 found
   Marina             10 found
   Ski               

KeyboardInterrupt: 